# Regularization

**IMSE 774 · Neural Networks · Week 9 · Session 17 · October 20, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/09-regularization.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

# The simplified network of Prince's figure 8.4, carried over from Week 8:
# a column of ones for the bias plus D slope-one ReLUs with joints at 0, 1/D, ..., (D-1)/D.
SIGMA = 0.15

def mu(x):
    return 0.5 * (np.exp(np.sin(2 * np.pi * x)) - 1.27)

def sample(I, rng):
    x = (np.arange(I) + rng.uniform(size=I)) / I
    return x, mu(x) + SIGMA * rng.standard_normal(I)

def features(x, D):
    joints = np.arange(D) / D
    return np.hstack([np.ones((len(x), 1)), np.maximum(x[:, None] - joints, 0)])

x_grid = np.linspace(0, 1, 501)

## L2 regularization


`fig-l2-sweep`


In [ ]:
def fit_l2(x, y, D, lam):
    """Equation (9.5) in closed form for the simplified model; the bias is not penalized."""
    X = features(x, D)
    P = np.eye(D + 1); P[0, 0] = 0
    return np.linalg.solve(X.T @ X + lam * P, X.T @ y)

I, D = 15, 14
rng = np.random.default_rng(774)
datasets = [sample(I, rng) for _ in range(200)]
lams = np.logspace(-6, 1, 29)
err = np.array([[np.mean((features(x_grid, D) @ fit_l2(x, y, D, lam) - mu(x_grid)) ** 2) for x, y in datasets]
                for lam in lams])
med = np.median(err, axis=1); q1, q3 = np.percentile(err, [25, 75], axis=1)

fig, ax = plt.subplots(1, 2, figsize=(8.5, 3.3))
ax[0].fill_between(lams, q1, q3, color=NDSU_GREEN, alpha=0.2, lw=0)
ax[0].plot(lams, med, color=NDSU_GREEN)
ax[0].plot(lams[med.argmin()], med.min(), "o", color=NDSU_GREEN)
ax[0].set(xscale="log", yscale="log", xlabel="$\\lambda$", ylabel="Error vs. true function")

x, y = datasets[0]
ax[1].plot(x_grid, mu(x_grid), "k", lw=2, label="true $\\mu[x]$")
for lam, c in [(1e-6, ACCENT), (3e-3, NDSU_GREEN), (1.0, STEEL)]:
    phi = fit_l2(x, y, D, lam)
    ax[1].plot(x_grid, features(x_grid, D) @ phi, color=c, lw=1.4,
               label=f"$\\lambda$ = {lam:g}, $\\|\\mathbf{{w}}\\|^2$ = {np.sum(phi[1:] ** 2):.3g}")
ax[1].plot(x, y, "o", color=NDSU_GOLD, mec="k", ms=5, zorder=5)
ax[1].set(ylim=(-1.2, 1.2), xlabel="Input, $x$", ylabel="Output, $y$")
ax[1].legend(fontsize=7.5, loc="lower left")
fig.tight_layout()
plt.show()

print(f"best lambda {lams[med.argmin()]:.1e}: error {med.min():.4f};  lambda -> 0: {med[0]:.4f};  lambda = 10: {med[-1]:.4f}")

## Implicit regularization in gradient descent


`fig-implicit-gd`


In [ ]:
def L_toy(p):  return (p[1] - 0.61) ** 2 * (1 + p[0] ** 2)
def g_toy(p):  return np.array([2 * p[0] * (p[1] - 0.61) ** 2, 2 * (p[1] - 0.61) * (1 + p[0] ** 2)])
def H_toy(p):
    a = p[1] - 0.61
    return np.array([[2 * a ** 2, 4 * p[0] * a], [4 * p[0] * a, 2 * (1 + p[0] ** 2)]])

def descend(grad, alpha, n, p0=(-1.5, -1.0)):
    p, path = np.array(p0, float), [np.array(p0, float)]
    for _ in range(n):
        p = p - alpha * grad(p); path.append(p.copy())
    return np.array(path)

flow = descend(g_toy, 5e-4, 60000)                                  # gradient flow, equation (9.6)
rows = []
for alpha in (0.02, 0.05, 0.1, 0.2):
    disc = descend(g_toy, alpha, int(60 / alpha))                    # gradient descent, equation (9.7)
    g_mod = lambda p, a=alpha: g_toy(p) + (a / 2) * H_toy(p) @ g_toy(p)   # gradient of L + (a/4) ||dL/dphi||^2
    mod = descend(g_mod, 5e-4, 60000)                                # gradient flow on equation (9.8)
    rows.append((alpha, disc, mod, (mod[-1, 0] - flow[-1, 0]) / (disc[-1, 0] - flow[-1, 0])))

P0, P1 = np.meshgrid(np.linspace(-1.8, 0.4, 200), np.linspace(-1.2, 1.0, 200))
LL = np.log(L_toy([P0, P1]) + 1e-3)
alpha, disc, mod, _ = rows[1]
fig, ax = plt.subplots(1, 2, figsize=(8.5, 3.6), sharey=True)
for a in ax:
    a.contourf(P0, P1, LL, levels=24, cmap="Greys_r", alpha=0.75)
    a.axhline(0.61, ls="--", color="gray", lw=1)
    a.plot(flow[:, 0], flow[:, 1], "--", color="k", lw=1.2, label="gradient flow")
    a.set(xlabel="$\\phi_0$")
ax[0].plot(disc[:, 0], disc[:, 1], "o-", color=NDSU_GOLD, mec="k", ms=3, lw=0.8, label=f"gradient descent, $\\alpha$ = {alpha}")
ax[1].plot(mod[:, 0], mod[:, 1], "-", color=NDSU_GREEN, lw=2, label="flow on the modified loss")
ax[1].plot(disc[-1, 0], disc[-1, 1], "o", color=NDSU_GOLD, mec="k", ms=6, label="where descent ended")
ax[0].set(ylabel="$\\phi_1$", title="Original loss"); ax[1].set(title="Loss + $\\frac{\\alpha}{4}\\|\\partial L/\\partial \\phi\\|^2$")
ax[0].legend(fontsize=8, loc="lower right"); ax[1].legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

print(f"{'alpha':>6}{'descent ends at':>17}{'flow ends at':>14}{'modified flow':>15}{'deviation explained':>21}")
for alpha, disc, mod, frac in rows:
    print(f"{alpha:>6}{disc[-1, 0]:>17.4f}{flow[-1, 0]:>14.4f}{mod[-1, 0]:>15.4f}{frac:>20.0%}")

## Early stopping


`fig-early-stopping`


In [ ]:
rng = np.random.default_rng(774)
x, y = sample(I, rng)
X, Xg = features(x, D), features(x_grid, D)
s, V = np.linalg.eigh(X.T @ X)                     # X^T X = V diag(s) V^T
c = V.T @ (X.T @ y)

def phi_at_time(t):
    """Gradient flow on sum_i (f - y_i)^2 from phi(0) = 0: each eigen-component approaches
    its least squares value with rate 2 s_k, so phi(t) = V diag((1 - exp(-2 s t)) / s) V^T X^T y."""
    return V @ ((1 - np.exp(-2 * s * t)) / s * c)

def phi_ridge(lam):
    """L2 regularization on every parameter: phi = V diag(1 / (s + lam)) V^T X^T y."""
    return V @ (c / (s + lam))

ts = np.logspace(-2, 5, 141)
test_t  = np.array([np.mean((Xg @ phi_at_time(t) - mu(x_grid)) ** 2) for t in ts])
train_t = np.array([np.mean((X  @ phi_at_time(t) - y) ** 2) for t in ts])
test_l  = np.array([np.mean((Xg @ phi_ridge(1 / (2 * t)) - mu(x_grid)) ** 2) for t in ts])

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(ts, test_t, color=NDSU_GREEN, lw=2, label="early stopping at time $t$: error vs. truth")
ax.plot(ts, train_t, "--", color=NDSU_GREEN, lw=1, label="early stopping: training error")
ax.plot(ts, test_l, color=STEEL, lw=2, label="L2 with $\\lambda = 1/(2t)$: error vs. truth")
ax.plot(ts[test_t.argmin()], test_t.min(), "o", color=NDSU_GREEN)
ax.set(xscale="log", yscale="log", xlabel="training time $t$ of gradient flow", ylabel="Mean squared error", ylim=(1e-4, 1))
top = ax.secondary_xaxis("top", functions=(lambda t: 1 / (2 * t), lambda l: 1 / (2 * l)))
top.set_xlabel("$\\lambda$")
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

print(f"best stopping time t = {ts[test_t.argmin()]:.0f}: error {test_t.min():.4f}, training error there {train_t[test_t.argmin()]:.4f}")
print(f"trained to convergence:  error {test_t[-1]:.4f}, training error {train_t[-1]:.1e}")

## Ensembling


`bagging`


In [ ]:
rng = np.random.default_rng(774)
single, bagged = [], []
for _ in range(200):
    x, y = sample(I, rng)
    f_single = Xg @ fit_l2(x, y, D, 1e-4)                            # one model on all the data
    f_bag = np.mean([Xg @ fit_l2(x[idx], y[idx], D, 1e-4)             # 20 models on bootstrap resamples
                     for idx in (rng.integers(0, I, I) for _ in range(20))], axis=0)
    single.append(np.mean((f_single - mu(x_grid)) ** 2)); bagged.append(np.mean((f_bag - mu(x_grid)) ** 2))
print(f"error vs. true function, median over 200 training sets:  single model {np.median(single):.4f}   bagged ensemble of 20 {np.median(bagged):.4f}")
print(f"the ensemble is better on {np.mean(np.array(bagged) < np.array(single)):.0%} of the training sets")

unique_frac = np.mean([len(np.unique(rng.integers(0, I, I))) / I for _ in range(2000)])
print(f"a bootstrap resample of {I} points contains on average {unique_frac:.0%} distinct points (the limit is 1 - 1/e = {1 - np.exp(-1):.0%})")

## Dropout


`fig-dropout`


In [ ]:
rng = np.random.default_rng(774)
x, y = sample(15, rng)
X_t = torch.tensor(x, dtype=torch.float32)[:, None]; Y_t = torch.tensor(y, dtype=torch.float32)[:, None]
Xg_t = torch.tensor(x_grid, dtype=torch.float32)[:, None]

def dropout_net(p):
    return nn.Sequential(nn.Linear(1, 100), nn.ReLU(), nn.Dropout(p),
                         nn.Linear(100, 100), nn.ReLU(), nn.Dropout(p),
                         nn.Linear(100, 1))

fig, ax = plt.subplots(1, 2, figsize=(8.5, 3.3), sharey=True)
results = {}
for a, p in zip(ax, (0.0, 0.5)):
    torch.manual_seed(774)
    net = dropout_net(p)
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for _ in range(4000):
        opt.zero_grad(); loss = ((net(X_t) - Y_t) ** 2).mean(); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        f = net(Xg_t).squeeze().numpy()
    net.train()                                                      # Monte Carlo dropout: keep the masks on
    with torch.no_grad():
        mc = torch.stack([net(Xg_t).squeeze() for _ in range(30)]).numpy()
    results[p] = (np.mean((f - mu(x_grid)) ** 2), np.mean((mc.mean(0) - mu(x_grid)) ** 2))
    if p > 0:
        a.fill_between(x_grid, mc.mean(0) - 2 * mc.std(0), mc.mean(0) + 2 * mc.std(0),
                       color=NDSU_GREEN, alpha=0.18, lw=0, label="Monte Carlo dropout, $\\pm 2$ sd of 30 passes")
    a.plot(x_grid, mu(x_grid), "k", lw=2, label="true $\\mu[x]$")
    a.plot(x_grid, f, color=ACCENT if p == 0 else NDSU_GREEN, lw=2, label="network, eval mode")
    a.plot(x, y, "o", color=NDSU_GOLD, mec="k", ms=5, zorder=5)
    a.set(title=f"dropout p = {p}", xlabel="Input, $x$", ylim=(-1.2, 1.2))
ax[0].set(ylabel="Output, $y$"); ax[1].legend(fontsize=7.5, loc="upper right")
fig.tight_layout()
plt.show()

for p, (e_eval, e_mc) in results.items():
    print(f"p = {p}: error vs. truth {e_eval:.4f} (eval mode), {e_mc:.4f} (mean of 30 Monte Carlo passes)")

## Applying noise


`input-noise`


In [ ]:
rng = np.random.default_rng(774)
I_n, sigma_x = 20, 0.3
x_n = rng.uniform(-1, 1, I_n); y_n = 0.5 + 1.5 * x_n + rng.normal(0, 0.2, I_n)
phi = np.array([0.3, 0.9])                                               # any parameter setting

def loss_line(xs):
    return np.sum((phi[0] + phi[1] * xs - y_n) ** 2)

noisy = np.mean([loss_line(x_n + rng.normal(0, sigma_x, I_n)) for _ in range(100000)])
print(f"expected loss with input noise (Monte Carlo):   {noisy:.4f}")
print(f"L + I sigma_x^2 phi_1^2, equation (input noise): {loss_line(x_n) + I_n * sigma_x ** 2 * phi[1] ** 2:.4f}")

`label-smoothing`


In [ ]:
K, rho = 10, 0.1
target = np.full(K, rho / (K - 1)); target[0] = 1 - rho              # the smoothed target distribution

def smoothed_ce(gap):
    """Cross-entropy between the smoothed target and a softmax whose correct-class logit
    exceeds the K - 1 others (all equal) by `gap`."""
    z = np.zeros(K); z[0] = gap
    log_p = z - (z.max() + np.log(np.exp(z - z.max()).sum()))
    return -(target * log_p).sum()

gaps = np.linspace(0, 15, 15001)
losses = np.array([smoothed_ce(g) for g in gaps])
best = gaps[losses.argmin()]
entropy = -(target * np.log(target)).sum()

print(f"K = {K} classes, rho = {rho}: the smoothed cross-entropy is minimized at a logit gap of {best:.3f}")
print(f"closed form log[(1 - rho)(K - 1) / rho] = {np.log((1 - rho) * (K - 1) / rho):.3f}")
print(f"minimum loss {losses.min():.4f} = entropy of the target {entropy:.4f}  (not zero: the target is not one-hot)")
print(f"with rho = 0 the loss keeps falling as the gap grows: gap 15 gives {-np.log(1 / (1 + (K - 1) * np.exp(-15))):.2e}, gap 30 gives {-np.log(1 / (1 + (K - 1) * np.exp(-30))):.2e}")

## Transfer learning and multi-task learning


`transfer-head`


In [ ]:
import torchvision

net = torchvision.models.resnet18(weights=None)           # the architecture of Chapter 11; weights="DEFAULT" would load ImageNet pre-training
n_total = sum(p.numel() for p in net.parameters())
print(f"ResNet-18: {n_total:,} parameters; its last layer is {net.fc}")

for p in net.parameters():                                 # freeze everything that was pre-trained
    p.requires_grad = False
net.fc = nn.Linear(net.fc.in_features, 4)                   # a new head for a four-class inspection task
n_head = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"frozen backbone + new head: {n_head:,} trainable parameters ({n_head / n_total:.2%} of the model)")

for p in net.layer4.parameters():                           # fine-tune the last block as well
    p.requires_grad = True
n_ft = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"also fine-tuning the last residual block: {n_ft:,} trainable parameters")

## Four Mechanisms


`parameter-sharing`


In [ ]:
H, W, C = 224, 224, 3                                      # a small colour image
dense = H * W * C * 1000                                   # one fully connected layer to 1,000 hidden units
conv = 3 * 3 * C * 64 + 64                                 # one 3x3 convolution producing 64 channels, applied at every position
print(f"fully connected layer, every pixel to 1,000 units: {dense:>12,} weights")
print(f"3x3 convolution with 64 channels, shared everywhere: {conv:>12,} parameters")
print(f"ratio: {dense / conv:,.0f} to 1")